# IT3212 - Assignment 3 - Task 3: Feature Extraction

Adds **derived clinical features** to the Task 2 output.

Task 2 produced 22 cleaned features. Task 3 **adds** 9 new columns computed
from existing ones. Nothing is removed - the source columns are kept, and
deciding which columns to keep is Task 4's job.

**Why extract features at all:** a model can only split on one column at a
time. A relationship *between* two columns (e.g. weight relative to height)
is invisible to it unless we compute that relationship as its own column.

**Design principle:** only *established, named clinical indices* are added.
Each one can be justified from clinical practice and verified against
published definitions. Ad-hoc combinations are deliberately excluded.

**Inputs:**  `../data/assignment3/task2_{train,val,test}.csv`

**Outputs:** `../data/assignment3/task3_{train,val,test}.csv`
             `../data/assignment3/task3_metadata.json`

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load the Task 2 output

All three splits are loaded. The target column is `smoking` (0 = never,
1 = ever smoked), as defined in Task 2.

In [2]:
IN_DIR = Path("../data/assignment3")
OUT_DIR = Path("../data/assignment3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "smoking"

splits = {
    name: pd.read_csv(IN_DIR / f"task2_{name}.csv")
    for name in ["train", "val", "test"]
}

for name, d in splits.items():
    print(f"task2_{name}.csv: {d.shape[0]:>7,} rows x {d.shape[1]} columns")

task2_train.csv: 787,625 rows x 23 columns
task2_val.csv:  98,453 rows x 23 columns
task2_test.csv:  98,454 rows x 23 columns


## 2. Why this extraction method

Feature extraction can be done in several ways, so the choice needs justifying
before the features themselves are listed.

**Alternatives considered:**

| Method | Why it was not used here |
|---|---|
| PCA | Unsupervised - it maximises variance in the features, not their relationship with the target. On a supervised problem it can discard exactly the directions that separate the classes |
| PLS | Supervised and a reasonable choice, but it produces abstract components that are linear combinations of the inputs. Those cannot be named, explained, or traced back to a clinical measurement |
| Polynomial / interaction expansion | Generates hundreds of columns, most of them noise, and inflates dimensionality faster than Task 4 can prune it |
| Binning (age bands, BP categories) | Discards information; tree-based models already find their own thresholds |

**Chosen approach: established clinical indices.** Each added feature is a
named measure already used in clinical practice or research.

This was chosen for three reasons:

1. **A model splits on one column at a time.** A relationship *between* two
   columns - weight relative to height, cholesterol relative to HDL - is
   invisible to it unless that relationship is computed as its own column.
   These indices make such relationships directly available.
2. **They are interpretable.** Every feature can be named and explained, which
   matters for Tasks 6, 8 and 10 where results have to be discussed rather than
   only tabulated. PLS components would not allow this.
3. **They are verifiable.** An established index can be checked against a
   published definition when citing sources. An invented ratio cannot.

**Reduction is deliberately left to Task 4.** This task expands the feature
space; Task 4 decides which columns to keep. Keeping the two steps separate
means the selection step has something real to select from, and its effect can
be measured.

## 3. The extracted features

Nine established clinical indices. Each is a named measure used in clinical
practice or research, so each can be justified individually.

| Feature | Formula | Established as |
|---|---|---|
| `BMI` | weight / (height/100)^2 | Body mass index - universal clinical measure |
| `waist_height_ratio` | waistline / height | Central adiposity - WHO standard |
| `pulse_pressure` | SBP - DBP | Arterial stiffness |
| `MAP` | DBP + (SBP - DBP)/3 | Mean arterial pressure - standard in critical care |
| `non_HDL_chole` | tot_chole - HDL_chole | Atherogenic cholesterol burden |
| `cholesterol_ratio` | tot_chole / HDL_chole | Cardiovascular risk ratio |
| `LDL_HDL_ratio` | LDL_chole / HDL_chole | Lipid risk ratio |
| `TG_HDL_ratio` | triglyceride / HDL_chole | Insulin resistance / metabolic syndrome |
| `AST_ALT_ratio` | SGOT_AST / SGOT_ALT | De Ritis ratio - liver damage pattern |

**Excluded:** `gamma_GTP / SGOT_AST` was considered but rejected - it is an
ad-hoc combination with no established definition, so it cannot be justified
or verified against a reference.

**Note on `AST_ALT_ratio`:** the De Ritis ratio is classically used to
distinguish alcohol-related liver damage from other causes. It is a
*measured* marker, not a self-report, so the circularity argument that led us
to exclude `DRK_YN` does not apply. It is included, with the alcohol overlap
acknowledged in the report.

**All nine features are stateless.** Each row's output depends only on that
row's inputs, with no parameters learned from the data. They can therefore be
applied identically to train, validation and test without any leakage risk.

## 4. Extraction function

All nine features are **stateless**: each row's output depends only on that
row's inputs, with no parameters learned from the data. They can therefore be
applied identically to train, validation and test without any leakage risk.

A **division guard** is applied to every ratio. The divisor minima in this
dataset are all >= 1.0 (`HDL_chole` >= 16, `SGOT_ALT` >= 1.0,
`height` >= 132.5), so division by zero cannot occur here. The guard is
included anyway so the code is safe if the data changes, and so the
assumption is explicit rather than implicit.

In [3]:
def safe_divide(numerator, denominator):
    """Element-wise division that returns NaN instead of inf where the
    denominator is zero or negative."""
    return np.where(denominator > 0, numerator / denominator, np.nan)


def extract_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with the nine clinical indices appended."""
    out = df.copy()

    height_m = safe_divide(out["height"], pd.Series(100.0, index=out.index))

    # Anthropometric
    out["BMI"] = safe_divide(out["weight"], height_m**2)
    out["waist_height_ratio"] = safe_divide(out["waistline"], out["height"])

    # Blood pressure
    out["pulse_pressure"] = out["SBP"] - out["DBP"]
    out["MAP"] = out["DBP"] + (out["SBP"] - out["DBP"]) / 3.0

    # Lipid profile
    out["non_HDL_chole"] = out["tot_chole"] - out["HDL_chole"]
    out["cholesterol_ratio"] = safe_divide(out["tot_chole"], out["HDL_chole"])
    out["LDL_HDL_ratio"] = safe_divide(out["LDL_chole"], out["HDL_chole"])
    out["TG_HDL_ratio"] = safe_divide(out["triglyceride"], out["HDL_chole"])

    # Liver enzymes
    out["AST_ALT_ratio"] = safe_divide(out["SGOT_AST"], out["SGOT_ALT"])

    return out


NEW_FEATURES = [
    "BMI",
    "waist_height_ratio",
    "pulse_pressure",
    "MAP",
    "non_HDL_chole",
    "cholesterol_ratio",
    "LDL_HDL_ratio",
    "TG_HDL_ratio",
    "AST_ALT_ratio",
]

## 5. Apply to all three splits

The same function is applied to train, validation and test. Because the
features are stateless, no fitting is involved and the three sets remain
directly comparable.

In [4]:
for name, d in splits.items():
    before = d.shape[1]
    splits[name] = extract_features(d)
    after = splits[name].shape[1]
    print(f"{name:5s}: {before} -> {after} columns (+{after - before})")

train: 23 -> 32 columns (+9)
val  : 23 -> 32 columns (+9)
test : 23 -> 32 columns (+9)


## 6. Sanity checks

Confirm that the extraction behaved as intended:

- no infinities or NaNs introduced
- the target column is unchanged
- source columns are still present
- derived values fall in plausible clinical ranges

In [5]:
for name, d in splits.items():
    n_nan = int(d[NEW_FEATURES].isnull().sum().sum())
    n_inf = int(np.isinf(d[NEW_FEATURES].to_numpy()).sum())
    print(f"{name:5s}: NaNs in new features = {n_nan}, infinities = {n_inf}")

print("\nTarget unchanged:")
for name, d in splits.items():
    print(f"  {name:5s}: ever ratio = {d[TARGET].mean():.4f}")

print("\nSource columns still present:")
for col in ["height", "weight", "waistline", "SBP", "DBP", "HDL_chole", "SGOT_AST"]:
    print(f"  {col:15s}: {'yes' if col in splits['train'].columns else 'NO'}")

train: NaNs in new features = 0, infinities = 0
val  : NaNs in new features = 0, infinities = 0
test : NaNs in new features = 0, infinities = 0

Target unchanged:
  train: ever ratio = 0.3920
  val  : ever ratio = 0.3920
  test : ever ratio = 0.3920

Source columns still present:
  height         : yes
  weight         : yes
  waistline      : yes
  SBP            : yes
  DBP            : yes
  HDL_chole      : yes
  SGOT_AST       : yes


In [6]:
summary = splits["train"][NEW_FEATURES].describe().T[["mean", "min", "max"]]
summary.round(3)

,mean,min,max
BMI,23.874,13.528,40.000
waist_height_ratio,0.501,0.300,0.778
pulse_pressure,46.311,4.000,107.500
MAP,91.394,62.500,119.833
non_HDL_chole,138.501,5.000,278.000
cholesterol_ratio,3.637,1.056,18.375
LDL_HDL_ratio,2.120,0.208,12.750
TG_HDL_ratio,2.491,0.013,18.000
AST_ALT_ratio,1.187,0.110,41.500


## 7. Round the derived features (6 significant figures)

The derived features are ratios and divisions, so they carry long floating-point
artifacts (`95.333333333333314`, `0.5393939393939394`). These digits are not
measurement precision - they are binary representation noise.

**Rounding rule: 6 significant figures.** Significant figures are used instead
of a fixed number of decimal places because the features span very different
magnitudes (`TG_HDL_ratio` ~0.01, `non_HDL_chole` ~270). A fixed decimal rule
would either destroy resolution in the small-valued features or waste space on
the large ones. Significant figures adapt to magnitude automatically, with a
single uniform rule.

**Effect on information retention** (share of distinct values kept, train set):

| Feature | 2 dp | 4 dp | 6 sig-fig |
|---|---|---|---|
| `BMI` | 97.0% | 99.2% | 99.2% |
| `waist_height_ratio` | 1.1% | 62.5% | 98.5% |
| `pulse_pressure` | 100% | 100% | 100% |
| `MAP` | 94.0% | 94.0% | 94.0% |
| `non_HDL_chole` | 100% | 100% | 100% |
| `cholesterol_ratio` | 10.0% | 100% | 100% |
| `LDL_HDL_ratio` | 8.1% | 100% | 100% |
| `TG_HDL_ratio` | 8.7% | 100% | 100% |
| `AST_ALT_ratio` | 36.3% | 100% | 100% |
| **Average** | **50.6%** | **95.1%** | **99.1%** |

Six significant figures retains 99.1% of the distinct values on average, while
reducing the exported file from 177 MB to 131 MB. Rounding to 7 figures gives
identical retention at a larger size, so 6 is the point of diminishing returns.

Note: leading zeros are not significant. A value such as `0.000606` has only
three significant figures and is therefore unchanged by this rule.

In [7]:
SIG_FIGS = 6


def round_sig(x, n=SIG_FIGS):
    """Round a scalar to n significant figures.

    Leading zeros are not significant, so 0.000606 (3 sig-fig) is unchanged.
    Zero and non-finite values are returned as-is.
    """
    if x == 0 or not np.isfinite(x):
        return x
    return float(f"%.{n - 1}e" % x)


before_unique = {c: splits["train"][c].nunique() for c in NEW_FEATURES}

for name, d in splits.items():
    for col in NEW_FEATURES:
        d[col] = d[col].map(round_sig)

print("Distinct values retained (train):")
for c in NEW_FEATURES:
    after = splits["train"][c].nunique()
    before = before_unique[c]
    print(f"  {c:20s} {before:>7,} -> {after:>7,}  ({after / before * 100:5.1f}%)")

print(f"\nRounded to {SIG_FIGS} significant figures.")

Distinct values retained (train):
  BMI                      133 ->     132  ( 99.2%)
  waist_height_ratio     4,386 ->   4,297  ( 98.0%)
  pulse_pressure           159 ->     159  (100.0%)
  MAP                      250 ->     235  ( 94.0%)
  non_HDL_chole            261 ->     261  (100.0%)
  cholesterol_ratio     10,318 ->  10,318  (100.0%)
  LDL_HDL_ratio          9,074 ->   9,074  (100.0%)
  TG_HDL_ratio          12,865 ->  12,865  (100.0%)
  AST_ALT_ratio          1,086 ->   1,086  (100.0%)

Rounded to 6 significant figures.


## 8. Export the extended splits

The data is exported **unscaled**, as in Task 2. Each downstream model decides
whether to scale: tree-based models do not need it, while KNN, SVM and neural
networks do.

**No scaler file is shipped.** `task3_metadata.json` records the scaling recipe
instead - which columns to scale, with which method, fitted on the training set
only. This follows the same convention as Task 2 and keeps the pipeline free of
binary artifacts that would need to be regenerated whenever the feature set
changes (as it will again in Task 4).

Binary and ordinal columns (`sex`, `hear_left`, `hear_right`, `urine_protein`)
are excluded from scaling, consistent with Task 2.

`task3_metadata.json` also records the added features, their formulas, and the
rounding rule.

In [8]:
# --- Scaling recipe (no scaler is fitted or saved) --------------------------
# Downstream tasks fit a StandardScaler on task3_train.csv using exactly these
# columns, on the training set only. Binary/ordinal columns are excluded,
# following the same convention as Task 2.
BINARY_OR_ORDINAL = ["sex", "hear_left", "hear_right", "urine_protein"]
SCALE_COLS = [
    c for c in splits["train"].columns
    if c != TARGET and c not in BINARY_OR_ORDINAL
]

print(f"{len(SCALE_COLS)} continuous columns (scaling recipe recorded in metadata.json)")

# --- Export the splits ------------------------------------------------------
for name, d in splits.items():
    d.to_csv(OUT_DIR / f"task3_{name}.csv", index=False)

metadata = {
    "task": "task3_feature_extraction",
    "target": TARGET,
    "target_classes": {"0": "never smoked", "1": "ever smoked (current or former)"},
    "source": "task2 output",
    "added_features": NEW_FEATURES,
    "feature_definitions": {
        "BMI": "weight / (height/100)^2",
        "waist_height_ratio": "waistline / height",
        "pulse_pressure": "SBP - DBP",
        "MAP": "DBP + (SBP - DBP)/3",
        "non_HDL_chole": "tot_chole - HDL_chole",
        "cholesterol_ratio": "tot_chole / HDL_chole",
        "LDL_HDL_ratio": "LDL_chole / HDL_chole",
        "TG_HDL_ratio": "triglyceride / HDL_chole",
        "AST_ALT_ratio": "SGOT_AST / SGOT_ALT",
    },
    "excluded_candidates": {
        "gamma_GTP_per_SGOT_AST": "ad-hoc combination with no established definition",
    },
    "feature_columns": [c for c in splits["train"].columns if c != TARGET],
    "n_features": len([c for c in splits["train"].columns if c != TARGET]),
    "scaling": {
        "applied_to_exports": False,
        "method": "StandardScaler",
        "columns": SCALE_COLS,
        "excluded_columns": BINARY_OR_ORDINAL,
        "fit_on": "train only - never fit on train+val+test",
        "note": (
            "No scaler file is shipped, following the same convention as Task 2. "
            "Downstream tasks fit a StandardScaler on task3_train.csv using "
            "exactly these columns."
        ),
    },
    "rounding": {
        "rule": "6 significant figures",
        "applies_to": NEW_FEATURES,
        "rationale": (
            "Removes floating-point artifacts while preserving the resolution "
            "of the underlying measurements. Retains ~99.1% of distinct values."
        ),
    },
    "row_counts": {name: len(d) for name, d in splits.items()},
}

with open(OUT_DIR / "task3_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nExported:")
for f in sorted(OUT_DIR.glob("task3_*")):
    print(f"  {f.name}")

27 continuous columns (scaling recipe recorded in metadata.json)



Exported:
  task3_metadata.json
  task3_test.csv
  task3_train.csv
  task3_val.csv


## 9. Verify the exports

Reload the files to confirm they are readable, carry the expected column count,
and preserve the target distribution.

In [9]:
for name in ["train", "val", "test"]:
    d = pd.read_csv(OUT_DIR / f"task3_{name}.csv")
    print(
        f"task3_{name}.csv: {d.shape[0]:>7,} rows x {d.shape[1]} columns "
        f"| ever ratio = {d[TARGET].mean():.4f}"
    )

print(f"\nFeatures: {metadata['n_features']} (+{len(NEW_FEATURES)} extracted)")
print(f"Continuous columns for downstream scaling: {len(SCALE_COLS)}")
print(f"Excluded from scaling: {BINARY_OR_ORDINAL}")

pd.read_csv(OUT_DIR / "task3_train.csv")[[TARGET] + NEW_FEATURES].head()

task3_train.csv: 787,625 rows x 32 columns | ever ratio = 0.3920
task3_val.csv:  98,453 rows x 32 columns | ever ratio = 0.3920


task3_test.csv:  98,454 rows x 32 columns | ever ratio = 0.3920

Features: 31 (+9 extracted)
Continuous columns for downstream scaling: 27
Excluded from scaling: ['sex', 'hear_left', 'hear_right', 'urine_protein']


,smoking,BMI,waist_height_ratio,pulse_pressure,MAP,non_HDL_chole,cholesterol_ratio,LDL_HDL_ratio,TG_HDL_ratio,AST_ALT_ratio
0,0,23.8751,0.539394,64.0,95.3333,119.0,2.85938,1.43750,2.10938,1.125000
1,1,20.7612,0.458235,20.0,86.6667,126.0,4.00000,2.16667,4.11905,0.830000
2,0,20.8117,0.516129,30.0,80.0000,142.0,3.18462,1.50769,3.36923,0.923077
3,0,22.9592,0.585714,65.0,95.6667,109.0,4.02778,2.27778,3.77778,2.058820
4,0,21.4844,0.517500,26.0,94.6667,139.0,3.52727,2.05455,2.34545,1.500000
